WaveNet

In [1]:
import json
import numpy as np
import soundfile as sf

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

from wavenet import WaveNetModel, mu_law_decode

from IPython.display import Audio


Instructions for updating:
non-resource variables are not supported in the long term


In [2]:
SAMPLES = 16000
TEMPERATURE = 1.0
WAVENET_PARAMS = './wavenet_params_default.json'
WAV_OUT = "generated.wav"

In [3]:
def generate_audio(model_dir, wavenet_params):
    net = WaveNetModel(
        batch_size=1,
        dilations=wavenet_params['dilations'],
        filter_width=wavenet_params['filter_width'],
        residual_channels=wavenet_params['residual_channels'],
        dilation_channels=wavenet_params['dilation_channels'],
        quantization_channels=wavenet_params['quantization_channels'],
        skip_channels=wavenet_params['skip_channels'],
        use_biases=wavenet_params['use_biases'],
        scalar_input=wavenet_params['scalar_input'],
        initial_filter_width=wavenet_params['initial_filter_width'],
        global_condition_channels=None,
        global_condition_cardinality=None
    )

    samples = tf.placeholder(tf.int32)

    next_sample = net.predict_proba(samples)

    decode = mu_law_decode(
        samples,
        wavenet_params["quantization_channels"]
    )

    sess = tf.Session()

    variables_to_restore = {
        var.name[:-2]: var for var in tf.global_variables()
        if not ('state_buffer' in var.name or 'pointer' in var.name)
    }

    saver = tf.train.Saver(variables_to_restore)
    saver.restore(sess, model_dir)

    print("Model załadowany")

    quantization_channels = wavenet_params["quantization_channels"]

    waveform = (
        [quantization_channels // 2]
        * (net.receptive_field - 1)
    )

    waveform.append(
        np.random.randint(quantization_channels)
    )

    for step in range(SAMPLES):
        if len(waveform) > net.receptive_field:
            window = waveform[-net.receptive_field:]
        else:
            window = waveform
        outputs = [next_sample]

        prediction = sess.run(outputs, feed_dict={samples: window})[0]

        scaled_prediction = np.log(prediction) / TEMPERATURE
        scaled_prediction = (scaled_prediction -
                                np.logaddexp.reduce(scaled_prediction))
        scaled_prediction = np.exp(scaled_prediction)
        if TEMPERATURE == 1.0:
                np.testing.assert_allclose(
                        prediction, scaled_prediction, atol=1e-5,
                        err_msg='Prediction scaling at temperature=1.0 '
                                'is not working as intended.')

        sample = np.random.choice(
            np.arange(quantization_channels),
            p=scaled_prediction
        )

        waveform.append(sample)

    audio = sess.run(
        decode,
        feed_dict={samples: waveform}
    )

    return audio
    

Defaultowe parametry (10k kroków)

In [4]:
with open(WAVENET_PARAMS, 'r') as config_file:
    wavenet_params = json.load(config_file)

In [5]:
model_dir = './logdir/train/2026-05-19T19-24-02/model.ckpt-9999'

audio = generate_audio(model_dir, wavenet_params)

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Deprecated in favor of operator or tf.math.divide.
Instructions for updating:
Use `tf.cast` instead.

INFO:tensorflow:Restoring parameters from ./logdir/train/2026-05-19T19-24-02/model.ckpt-9999
Model załadowany


In [6]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

Uproszczone parametry 

In [7]:
with open('./wavenet_params.json', 'r') as config_file_light:
    wavenet_params_light = json.load(config_file_light)

In [12]:
model_dir = './logdir/train/2026-05-19T14-46-50/model.ckpt-9999'

audio = generate_audio(model_dir, wavenet_params_light)

INFO:tensorflow:Restoring parameters from ./logdir/train/2026-05-19T14-46-50/model.ckpt-9999
Model załadowany


In [13]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

Spektralna funkcja straty

In [16]:
model_dir = './logdir/train/2026-05-20T16-01-29/model.ckpt-9999'

audio = generate_audio(model_dir, wavenet_params)

INFO:tensorflow:Restoring parameters from ./logdir/train/2026-05-20T16-01-29/model.ckpt-9999
Model załadowany


C:\Users\wojte\AppData\Local\Temp\ipykernel_4052\1476046494.py:58: RuntimeWarning: divide by zero encountered in log
  scaled_prediction = np.log(prediction) / TEMPERATURE


In [17]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

Mu-law (mu=63, quantization_channels=64)

In [35]:
wavenet_params["quantization_channels"] = 64

In [36]:
model_dir = './logdir/train/2026-05-21T16-19-16/model.ckpt-9999'

audio = generate_audio(model_dir, wavenet_params)

INFO:tensorflow:Restoring parameters from ./logdir/train/2026-05-21T16-19-16/model.ckpt-9999
Model załadowany


In [37]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

Mu-law (mu=511, quantization_channels=512)

In [40]:
wavenet_params["quantization_channels"] = 512

In [41]:
model_dir = './logdir/train/2026-05-22T14-10-50/model.ckpt-9999'

audio = generate_audio(model_dir, wavenet_params)

INFO:tensorflow:Restoring parameters from ./logdir/train/2026-05-22T14-10-50/model.ckpt-9999
Model załadowany


In [42]:
Audio(
    audio,
    rate=wavenet_params["sample_rate"]
)

In [39]:
tf.reset_default_graph()